# 10 · End-to-End Project

> **Module 5 · Capstone · Lecture 5.1**

In this notebook you will:

- Generate the sample dataset and understand exactly what it contains
- Run **every** DAG in the local runner, and cross-validate each output against the pure reference implementation
- Walk through the four real DAG files in `project/dags/` — what each task does and why it exists
- See precisely what changes when those DAGs are deployed to Composer
- Build the deployment checklist: Variables, Connections, DAG params, and how to trigger a backfill
- Prove the pipeline is idempotent, so a retry and a backfill are both safe

## 1. The project

Everything in the course has been building toward one pipeline. Now we run it end to end, check the numbers, and work out how to ship it.

The project is a **daily orders ETL**:

```
   GCS                        BigQuery                    GCS
gs://source/            →   analytics.orders_enriched  →  gs://target/
2026-01-15/orders.csv      (partitioned by date)         2026-01-15/summary.csv
```

Four DAGs implement it, four supporting modules hold the business logic, and one local runner executes them without Airflow.

### The file layout

```
project/
├── core/                     ← pure Python, no Airflow, fully testable
│   ├── pipeline_logic.py         parse, validate, aggregate, render
│   ├── templating.py             Jinja rendering, Airflow-style context
│   ├── fanout.py                 the fan-out / fan-in helpers
│   ├── sensor_conditions.py      the wait-for-data predicate
│   ├── idempotent_load.py        truncate vs append semantics
│   ├── dedup_load.py             business-key deduplication
│   ├── sla_logic.py              SLA computation
│   └── sla_tracker.py            SLA state tracking
├── dags/                     ← the artefact you deploy to Composer
│   ├── dag_config.py             shared, Variable-overridable config
│   ├── 01_hello_world.py
│   ├── 02_templating_and_variables.py
│   ├── 03_etl_bigquery.py
│   └── 04_branching_and_xcom.py
├── local_runner.py           ← the mini-Airflow
└── tests/
```

**The separation is the whole point.** `core/` imports nothing from Airflow; `dags/` imports Airflow and calls into `core/`. That is why the logic can be unit-tested in under a second on a machine with no Airflow install — and why the same functions serve a task, a sensor and a test without duplication.

### The two things you deploy

| | Where it lives | Deployed to Composer? |
|---|---|---|
| **Business logic** | `project/core/*.py` | Yes — as a Python package in the DAG bucket |
| **Orchestration** | `project/dags/*.py` | Yes — as DAG files in the DAG bucket |
| **The runner** | `project/local_runner.py` | **No** — it is a teaching tool |

> **The local runner is not what you deploy.** It simulates Airflow's semantics so you can verify the logic without an Airflow install. What you deploy is `dags/` plus `core/`.

### How to run this notebook

The cells below import from the course tree. If you are running Jupyter from the course root, the path preamble finds everything; if you are running from `notebooks/`, it walks up one level. Either way, no credentials and no network access are needed — every cell runs offline.

In [ ]:
"""The path preamble and the course tree.

Every notebook in this course starts here. It finds the course root no
matter which directory Jupyter was launched from, and puts the two
directories that matter on `sys.path`:

    project/core   — the pure business logic
    project        — the local runner
"""

from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print(f"COURSE_ROOT = {COURSE_ROOT}")
print()

# The modules this notebook uses. Importing them here — once — proves
# that they are importable with no Airflow installed and no credentials.
import pipeline_logic
import fanout
import sensor_conditions
import idempotent_load
import dedup_load
import templating
import local_runner

for module in (
    pipeline_logic,
    fanout,
    sensor_conditions,
    idempotent_load,
    dedup_load,
    templating,
    local_runner,
):
    print(f"  {module.__name__:<20} {Path(module.__file__).relative_to(COURSE_ROOT)}")

print()
print("Airflow is not installed and is not needed:")
print("  * core/*.py imports nothing from Airflow")
print("  * dags/*.py is the artefact you deploy; it is never imported here")
print()

## 2. Generate the sample data

The dataset is generated by `resources/sample_data.py`, which is **seeded** — every run produces byte-identical output. That is what makes the expected numbers in this notebook assertable.

```bash
# RUN IN A TERMINAL — not executed here.
# Regenerate the sample dataset from scratch.
python resources/sample_data.py --output-dir ./sample_data

# ... and upload it to the bucket the DAGs will read from.
python resources/sample_data.py --output-dir ./sample_data --upload gs://YOUR_PROJECT_ID-source
```

The generator writes four files:

| File | What it is |
|---|---|
| `orders.csv` | 5000 orders, one per line, with a header |
| `orders.ndjson` | The same orders as newline-delimited JSON |
| `malformed_orders.csv` | 8 deliberately broken rows, one per reject reason |
| `product_reference.ndjson` | 50 product reference rows, for a join |
| `summary.json` | A manifest of what was generated |

The default is 5000 orders dated **2026-01-15** — one calendar day, so the whole dataset falls inside a single half-open data interval.

Now let us generate it and look at it.

In [ ]:
"""Generate the sample dataset and inspect it.

The generator lives in `resources/sample_data.py`. We import its
functions directly so the notebook is self-contained — but the script
is the canonical entry point, and it is the same code either way.

Everything here is deterministic: the generator is seeded with 42, so
the dataset is byte-identical on every run and the expected numbers in
this notebook stay correct.
"""

import importlib.util
import json
import random
from collections import Counter
from datetime import date, datetime
from pathlib import Path

# --- import resources/sample_data.py without making it a package --------
_RESOURCE_PATH = COURSE_ROOT / "resources" / "sample_data.py"
_spec = importlib.util.spec_from_file_location("sample_data", _RESOURCE_PATH)
sample_data = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(sample_data)

SAMPLE_DIR = COURSE_ROOT / "sample_data"
TARGET_DATE = "2026-01-15"

print("=" * 78)
print("GENERATING THE SAMPLE DATA")
print("=" * 78)
print()

target = datetime.strptime(TARGET_DATE, "%Y-%m-%d").date()
rng = random.Random(sample_data.SEED)

orders = sample_data.generate_orders(5000, target, rng=rng)
malformed = sample_data.generate_malformed_rows(5)

print(f"orders generated       : {len(orders)}")
print(f"malformed rows         : {len(malformed)}")
print(f"target date            : {target.isoformat()}")
print(f"regions                : {sample_data.REGIONS}")
print(f"currencies             : {sample_data.CURRENCIES}")
print(f"statuses               : {sample_data.STATUSES}")
print()
print("The first three orders:")
for row in orders[:3]:
    print(f"  {row['order_id']}  {row['order_ts']}  {row['region']:<6} "
          f"{row['currency']}  {row['status']:<10} {row['amount']:>8.2f}")
print()

# --- the distribution, which is what the aggregation actually measures ---
print("=" * 78)
print("THE DISTRIBUTION")
print("=" * 78)
print()

region_counts = Counter(r["region"] for r in orders)
status_counts = Counter(r["status"] for r in orders)
currency_counts = Counter(r["currency"] for r in orders)

print("orders by region:")
for region in sorted(region_counts):
    print(f"  {region:<8} {region_counts[region]:>6}")
print()
print("orders by status:")
for status in sorted(status_counts):
    print(f"  {status:<10} {status_counts[status]:>6}")
print()
print("orders by currency:")
for currency in sorted(currency_counts):
    print(f"  {currency:<5} {currency_counts[currency]:>6}")
print()

# --- every timestamp falls inside one half-open interval ---------------
timestamps = [r["order_ts"] for r in orders]
print("timestamp range:")
print(f"  earliest : {min(timestamps)}")
print(f"  latest   : {max(timestamps)}")
print(f"  all on {target.isoformat()}: "
      f"{all(ts.startswith(target.isoformat()) for ts in timestamps)}")
print()
print("That matters: the whole dataset falls inside the data interval")
print(f"  [{target.isoformat()}, 2026-01-16)")
print("so the daily filter keeps every row, and the half-open boundary")
print("never splits a row between two days.")
print()

# --- the malformed rows: one per reject reason -------------------------
print("=" * 78)
print("THE MALFORMED ROWS")
print("=" * 78)
print()
for index, line in enumerate(malformed, start=1):
    print(f"  {index}. {line}")
print()
print("Each one triggers a different reject reason code. The dead-letter")
print("path is only testable if there is something to reject.")
print()

# --- write the files, exactly as the script does -----------------------
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
sample_data.write_csv(orders, SAMPLE_DIR / "orders.csv", sample_data.ORDER_FIELDS)
sample_data.write_ndjson(orders, SAMPLE_DIR / "orders.ndjson")
sample_data.write_text_lines(
    malformed, SAMPLE_DIR / "malformed_orders.csv"
)
sample_data.write_ndjson(
    list(sample_data.generate_reference_data().values()),
    SAMPLE_DIR / "product_reference.ndjson",
)

manifest = {
    "orders": len(orders),
    "malformed_rows": len(malformed),
    "reference_rows": len(sample_data.generate_reference_data()),
    "target_date": TARGET_DATE,
}
(SAMPLE_DIR / "summary.json").write_text(
    json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
)

print("files written:")
for path in sorted(SAMPLE_DIR.iterdir()):
    if path.is_file():
        print(f"  {path.name:<28} {path.stat().st_size:>9,} bytes")
print()
print("manifest:", manifest)
print()

## 3. Cross-validate against the pure reference implementation

The pipeline has two implementations of the same idea:

- **The pure one** — `pipeline_logic.transform_orders` and `pipeline_logic.summarise_by_region`, called directly.
- **The orchestrated one** — the same functions, called through the local runner's task graph.

They should agree exactly. If they do not, the orchestration is wrong — and that is the whole reason for having a local runner at all.

The reference implementation is also the *specification*: it is what the unit tests assert against, and it is the thing you read when you want to know what the pipeline is supposed to do.

### The expected result

For the default dataset, the region summary is exactly:

```
region,order_count,revenue_eur
AMER,318,113592.19
APAC,285,108194.92
EMEA,322,118423.22
LATAM,320,110646.18
```

and the row counts are:

```
accepted=1245 rejected=0
```

Note the arithmetic that makes this non-obvious: **5000 orders in, 1245 accepted.** The difference is the status filter — only `complete` orders count as revenue-bearing. The other 3755 rows are *accepted by the validator* and then excluded because they are `active`, `cancelled` or `refunded`. That distinction is not an accident; `transform_orders` documents it explicitly:

> Status filtering happens here, not in the validator, so that "invalid" and "not revenue-bearing" stay distinct.

`rejected=0` is also a deliberate choice: the clean `orders.csv` has no malformed rows. The malformed file is separate, so the happy path and the dead-letter path can be tested independently.

In [ ]:
"""The reference implementation, run directly.

This is the specification. The orchestrated run in the next cell must
produce byte-identical output.
"""

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    render_rejects_csv,
    reject_counts,
    logical_date_window,
    partition_for_date,
)

EXPECTED = {
    "AMER": {"order_count": 318, "revenue_eur": 113592.19},
    "APAC": {"order_count": 285, "revenue_eur": 108194.92},
    "EMEA": {"order_count": 322, "revenue_eur": 118423.22},
    "LATAM": {"order_count": 320, "revenue_eur": 110646.18},
}

raw_lines = (SAMPLE_DIR / "orders.csv").read_text(encoding="utf-8").splitlines()

print("=" * 78)
print("THE PURE REFERENCE IMPLEMENTATION")
print("=" * 78)
print()
print(f"raw lines read : {len(raw_lines)}")
print()

accepted, rejected = transform_orders(raw_lines)

print(f"accepted       : {len(accepted)}")
print(f"rejected       : {len(rejected)}")
print()

# Why the counts differ — the status filter.
status_counts = Counter(row["status"] for row in accepted)
print("accepted rows by status:")
for status in sorted(status_counts):
    print(f"  {status:<10} {status_counts[status]:>6}")
print()
print("Only 'complete' rows count as revenue-bearing. The others are")
print("valid rows that simply do not carry revenue, so they are")
print("accepted by the validator and then excluded from the sum.")
print()

summary = summarise_by_region(accepted)

print("=" * 78)
print("THE SUMMARY")
print("=" * 78)
print()
print(render_summary_csv(summary), end="")

print()
print("=" * 78)
print("CHECKING IT AGAINST THE EXPECTED RESULT")
print("=" * 78)
print()

assert summary == EXPECTED, f"reference summary changed:\n{summary}"
print(f"summary == EXPECTED            : True")
print(f"accepted count == 1245         : {len(accepted) == 1245}  ({len(accepted)})")
print(f"rejected count == 0            : {len(rejected) == 0}  ({len(rejected)})")
print()

# --- the interval the run covers ---------------------------------------
start, end = logical_date_window(TARGET_DATE)
partition = partition_for_date(TARGET_DATE)
print(f"data interval    : [{start}, {end})")
print(f"partition        : {partition}")
print()

# --- the dead-letter path, exercised separately ------------------------
print("=" * 78)
print("THE DEAD-LETTER PATH (malformed_orders.csv)")
print("=" * 78)
print()

bad_lines = (SAMPLE_DIR / "malformed_orders.csv").read_text(
    encoding="utf-8"
).splitlines()
bad_accepted, bad_rejected = transform_orders(bad_lines)

print(f"input rows       : {len(bad_lines)}")
print(f"accepted         : {len(bad_accepted)}")
print(f"rejected         : {len(bad_rejected)}")
print()
print("rejects by reason:")
for reason, count in sorted(reject_counts(bad_rejected).items()):
    print(f"  {reason:<24} {count:>4}")
print()

print("the rejects CSV:")
print(render_rejects_csv(bad_rejected), end="")

print()
print("PASS: the reference implementation produces the expected summary,")
print("      and the dead-letter path classifies every malformed row.")
print()

## 4. Run every DAG in the local runner

The runner knows four DAGs. Each one exercises a different part of the Airflow surface:

| DAG | Tasks | What it exercises |
|---|---|---|
| `etl_to_gcs` | 4 | The straight-line pipeline: read → transform → aggregate → write |
| `branching_load` | 6 | A `BranchOperator` decision and a `none_failed` join |
| `xcom_pipeline` | 5 | Passing values between tasks via XCom |
| `templating_paths` | 4 | Rendering GCS and BigQuery paths from Jinja templates |

The runner implements the subset of Airflow semantics that change *observable behaviour*: dependency ordering, upstream-failure propagation, trigger rules, branch fan-out and skip, retries with delay, and XCom push/pull. That is enough for a DAG defined for the runner to behave the way it will in Composer.

```bash
# RUN IN A TERMINAL — not executed here.
python project/local_runner.py --list
python project/local_runner.py --dag all --date 2026-01-15 --verbose
python project/local_runner.py --dag etl_to_gcs --describe
```

Let us run all four, in-process, and cross-check each against the reference.

In [ ]:
"""Run every DAG in the local runner and cross-validate each one.

For each DAG we check three things:

  1. every task reached a terminal state (nothing left `upstream_failed`),
  2. the DAG's summary equals the pure reference summary, and
  3. the DAG's row counts equal the reference row counts.

If any of these fail, the orchestration disagrees with the logic — which
is the bug class the local runner exists to catch.
"""

from local_runner import DAGS, run_dag

print("=" * 78)
print("THE DAGs THE RUNNER KNOWS")
print("=" * 78)
print()

for name in sorted(DAGS):
    dag = DAGS[name]()
    print(dag.describe())
    print()

# --------------------------------------------------------------------------
# Run them all
# --------------------------------------------------------------------------
print("=" * 78)
print("RUNNING ALL FOUR")
print("=" * 78)
print()

runs = {}

for name in sorted(DAGS):
    dag = DAGS[name]()
    print(f"▶ {name} @ {TARGET_DATE}")
    run = run_dag(dag, TARGET_DATE, verbose=True)
    runs[name] = run

    failed = [t for t, s in run.state.items() if s == "failed"]
    blocked = [t for t, s in run.state.items() if s == "upstream_failed"]
    skipped = [t for t, s in run.state.items() if s == "skipped"]

    print(f"  tasks: {len(dag.tasks)}   failed: {len(failed)}   "
          f"blocked: {len(blocked)}   skipped: {len(skipped)}")
    assert not failed, f"{name} failed: {failed}"
    assert not blocked, f"{name} had blocked tasks: {blocked}"
    print()

# --------------------------------------------------------------------------
# Cross-validate every DAG against the reference
# --------------------------------------------------------------------------
print("=" * 78)
print("CROSS-VALIDATION AGAINST THE REFERENCE")
print("=" * 78)
print()

reference_summary = summarise_by_region(accepted)
reference_accepted = len(accepted)
reference_rejected = len(rejected)

print(f"{'dag':<20} {'summary':<10} {'rows':<8} {'accept':>8} {'reject':>8}")
print("-" * 70)

for name in sorted(runs):
    run = runs[name]

    # Each DAG names its aggregate task differently; find the summary
    # wherever it was pushed. `branching_load` has no aggregate task at
    # all — it branches on the row count — so it is checked on its rows.
    summary = (
        run.pull("aggregate", "summary")
        or run.pull("publish_summary", "summary")
    )

    stats = (
        run.pull("transform_orders")
        or run.pull("validate_and_stage")
        or {}
    )
    accept_count = int(stats.get("accepted_count", 0))
    reject_count = int(stats.get("rejected_count", 0))

    summary_ok = summary is None or summary == reference_summary
    rows_ok = accept_count == reference_accepted

    summary_label = "-" if summary is None else (
        "MATCH" if summary == reference_summary else "DIFFER")

    print(f"{name:<20} {summary_label:<10} "
          f"{'MATCH' if rows_ok else 'DIFFER':<8} "
          f"{accept_count:>8} {reject_count:>8}")

    assert summary_ok, f"{name}: summary != reference\n{summary}\n{reference_summary}"
    assert rows_ok, f"{name}: accepted count != {reference_accepted}"
    assert reject_count == reference_rejected, f"{name}: rejected count != {reference_rejected}"

print()
print("Every DAG produced the reference summary, from the same input,")
print("through four different orchestration shapes.")
print()
print("`branching_load` shows '-' in the summary column because it has")
print("no aggregate task — it branches on the row count instead. Its")
print("row counts are still checked against the reference.")
print()

# --------------------------------------------------------------------------
# The summary they all agree on
# --------------------------------------------------------------------------
print("=" * 78)
print("THE OUTPUT")
print("=" * 78)
print()

final_summary = runs["etl_to_gcs"].pull("aggregate", "summary")
print(render_summary_csv(final_summary), end="")

stats = runs["etl_to_gcs"].pull("transform_orders")
print(f"accepted={stats['accepted_count']} rejected={stats['rejected_count']}")
print()

# The exact strings the course promises.
assert render_summary_csv(final_summary) == (
    "region,order_count,revenue_eur\n"
    "AMER,318,113592.19\n"
    "APAC,285,108194.92\n"
    "EMEA,322,118423.22\n"
    "LATAM,320,110646.18\n"
), render_summary_csv(final_summary)

assert stats["accepted_count"] == 1245
assert stats["rejected_count"] == 0

print("PASS: the output is exactly")
print()
print("  region,order_count,revenue_eur")
print("  AMER,318,113592.19")
print("  APAC,285,108194.92")
print("  EMEA,322,118423.22")
print("  LATAM,320,110646.18")
print()
print("  accepted=1245 rejected=0")
print()

# --------------------------------------------------------------------------
# The artefacts each DAG produced
# --------------------------------------------------------------------------
print("=" * 78)
print("THE ARTEFACTS")
print("=" * 78)
print()

etl_run = runs["etl_to_gcs"]
print(f"etl_to_gcs  wrote   : {etl_run.pull('write_summary', 'local_path')}")
print(f"            gcs uri : {etl_run.pull('write_summary', 'gcs_uri')}")
print()

xcom_run = runs["xcom_pipeline"]
print(f"xcom_pipeline pushed : "
      f"{sorted(xcom_run.xcom.all_for('publish_summary'))}")
print()

branch_run = runs["branching_load"]
print(f"branching_load chose : "
      f"{branch_run.pull('choose_strategy', 'branch')}")
print(f"            finalise : {branch_run.pull('finalise')}")
print()

## 5. Walk through the four DAG files

These are the files you deploy. Each is a real Airflow DAG: it imports `airflow`, uses real operators, and would parse in Composer without a change.

We read them as **text**, not by importing them — importing would require Airflow, and Airflow is deliberately absent from this machine. That is a feature, not a limitation: it proves the DAG files are genuinely deployable artefacts, and that everything worth *verifying* lives in `core/`, which is importable without Airflow.

### `01_hello_world.py` — the smallest useful DAG

**Purpose:** verify your Composer environment end to end. Upload it, wait for it to appear in the DAG list, trigger it, see the two tasks go green. If `hello_world` does not run, nothing else will.

**Tasks:**

| Task | Operator | What it does |
|---|---|---|
| `say_hello` | `PythonOperator` | Logs a greeting including the run's logical date |
| `print_date` | `BashOperator` | Echoes `{{ ds }}` and `{{ prev_ds }}` |

**The dependencies:** `greet >> print_date` — one operator, one line.

**What it teaches:**

- `{{ ds }}` is the logical date, not the wall clock. A run triggered at 09:00 on the 15th for the 14th's data has `ds == "2026-01-14"`.
- `prev_ds` is one interval earlier — the standard way to read "yesterday's" data relative to the run.
- A task's return value becomes its XCom. `say_hello` returns the greeting string, which you can pull downstream.

**On Composer:** it parses within 1–2 minutes of upload. If it does not appear in the DAG list, the import failed and the scheduler log has the traceback.

### `02_templating_and_variables.py` — every configuration source

**Purpose:** demonstrate the four places configuration can come from, and the rule that decides which one to use.

| Source | Scope | Use for |
|---|---|---|
| `params` | Per **run** | A backfill date, a flag, a row count |
| Variables | Per **environment** | Bucket names, project IDs, a threshold |
| Connections | Credentials | API hosts, logins, passwords |
| The DAG file | Logic and structure | The schedule, the task graph |

**Tasks:**

| Task | Operator | What it does |
|---|---|---|
| `report_environment` | `PythonOperator` | Logs the dag_id, task_id, logical_date, run_id, params, interval, and the owner Variable |
| `show_paths` | `BashOperator` | Renders five paths and echoes them |

**The details worth noticing:**

- `context["var"].get("pipeline_owner", "unassigned")` — a **default**. A missing Variable would otherwise fail the task at run time, in production, on a Saturday.
- `context["conn"].get("http_orders_api")` is wrapped in a `try/except` — a missing Connection is not fatal.
- The Connection's **host** is logged. Its password is not, and never should be.
- `show_paths` uses `{{ macros.ds_add(ds, 1) }}` — the macro system for date arithmetic.
- `{{ params.bucket | default(var.value.target_bucket) }}` — the layering in one expression: a param beats a Variable beats a hard-coded default.

**On Composer:** `params` are set when you trigger the DAG from the UI ("Trigger DAG w/ config"), or with `--conf` on the CLI. Variables and Connections are set in **Admin → Variables** and **Admin → Connections**.

### `03_etl_bigquery.py` — the production pipeline

**Purpose:** the main ETL. This is the DAG that ties the course together, and it is the shape most production Airflow pipelines have.

**The task graph:**

```
check_source_exists → validate_and_stage → load_to_bigquery
                                                  ↓
                              export_to_gcs ← aggregate_by_region
```

| Task | Operator | What it does |
|---|---|---|
| `check_source_exists` | `BashOperator` | Confirms the source object for `{{ ds }}` exists; echoes a warning rather than failing |
| `validate_and_stage` | `PythonOperator` | Returns the staging table, the partition, and the partition decorator |
| `load_to_bigquery` | `BigQueryInsertJobOperator` | Loads with `writeDisposition: WRITE_TRUNCATE` |
| `aggregate_by_region` | `BigQueryExecuteQueryOperator` | Aggregates by region over the half-open interval |
| `export_to_gcs` | `BigQueryToGCSOperator` | Exports the table to GCS under `{{ dag.dag_id }}/{{ ds }}` |

**Four decisions that make this correct in production:**

1. **`WRITE_TRUNCATE`, not `WRITE_APPEND`.** The load targets a partition decorator, so a retry replaces the partition instead of appending a second copy. This is the single most important line in the file.

2. **The half-open filter.** The aggregation says:

   ```sql
   WHERE order_ts >= '{{ data_interval_start }}'
     AND order_ts <  '{{ data_interval_end }}'
   ```

   A `<=` would double-count the row at exactly midnight; a `BETWEEN` would load that row into two days. This off-by-one is the most common bug in a daily Airflow pipeline and is invisible until the monthly totals are wrong.

3. **`depends_on_past=False`.** The DAG does not wait for yesterday to finish. Each day is independent, which is what makes a backfill safe and parallel.

4. **`max_active_runs=1`.** One run at a time, so a catchup storm cannot saturate the workers.

**The SLA.** `sla=timedelta(hours=2)`, measured from the **end of the data interval**, not from the wall-clock start. A 02:00 run whose interval ended at 00:00 has two hours of headroom regardless of when it started. `on_sla_miss` is the callback that fires when the SLA is breached.

**On Composer:** `BigQueryInsertJobOperator` and friends are Google provider operators — already installed in the Composer image. The `configuration` dict is rendered as a Jinja template, so `{{ ds }}` and the XCom pull resolve before the job is submitted.

### `04_branching_and_xcom.py` — decisions, sensors and chaining

**Purpose:** the runtime-decision DAG. It is manual-trigger only (`schedule=None`) because it takes parameters.

**The task graph:**

```
count_rows → wait_for_upstream_data → choose_strategy
                                              ├──→ full_load ──────┐
                                              └──→ incremental_load ┤
                                                                    ↓
                                                    load_complete → trigger_downstream_dag
```

| Task | Operator | What it does |
|---|---|---|
| `count_rows` | `PythonOperator` | Returns the row count; pushes the threshold to XCom |
| `wait_for_upstream_data` | `PythonSensor` | Polls until the expected row count is available |
| `choose_strategy` | `BranchOperator` | Returns `["full_load"]` or `["incremental_load"]` |
| `full_load` | `PythonOperator` | Rewrites the whole table |
| `incremental_load` | `PythonOperator` | Appends one partition |
| `load_complete` | `EmptyOperator` | The join point — `trigger_rule="none_failed"` |
| `trigger_downstream_dag` | `TriggerDagRunOperator` | Starts `branching_and_xcom_child` with a `conf` |

**Four things this file teaches:**

1. **Branching skips *downstream*, not itself.** `choose_strategy` returns a list of task_ids to run; every other downstream task is marked *skipped*.

2. **The join needs `none_failed`.** `load_complete` depends on *both* branches. One will be skipped, and the default `all_success` would skip `load_complete` too — a DAG that looks correct and quietly does nothing. `none_failed` means "run as long as nothing failed"; a skipped dependency is not a failure.

3. **The sensor releases its worker.** `PythonSensor(mode="reschedule")` frees the worker slot while it waits and is re-evaluated later. That is why a sensor is the right tool for "wait for X" and a `time.sleep` loop is the wrong one. `timeout=3600`, `poke_interval=300`.

4. **`TriggerDagRunOperator` chains DAGs.** The `conf` — `{"parent_run_id": "{{ run_id }}"}` — is the contract between parent and child. It must be JSON-serialisable; a `datetime` in it raises `TypeError`.

**On Composer:** trigger it from the UI with a config:

```json
{"row_count": 1500000, "force_full_load": false}
```

The `row_count` parameter is what drives the branch, and `force_full_load` overrides the threshold entirely — which is exactly the switch you want during an incident.

In [ ]:
"""Read the DAG files structurally, without importing Airflow.

Importing a DAG file would require Airflow, and Airflow is deliberately
absent from this machine. Instead we parse the *text*: pull out the
`dag_id`, the `schedule`, the operator classes, the task_ids and the
dependency edges.

This is a real technique, not a workaround. It is how you lint DAG
files in CI, count how many operators a DAG uses, and check that a DAG
has not silently grown a dependency that breaks a branch.
"""

import ast
import re
from pathlib import Path

DAGS_DIR = COURSE_ROOT / "project" / "dags"


def analyse_dag_file(path: Path) -> dict:
    """Extract the structure of a DAG file from its source text.

    Args:
        path: The path to the DAG file.

    Returns:
        A dict with the dag_id, schedule, operators, task_ids, params
        and the dependency edges.
    """
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    dag_id = None
    schedule = None
    start_date = None
    catchup = None
    max_active_runs = None
    sla = None
    tags: list[str] = []
    params: dict[str, object] = {}
    operators: list[str] = []
    task_ids: list[str] = []
    edges: list[tuple[str, str]] = []
    defaults: dict[str, object] = {}

    # --- the `with DAG(...)` call -----------------------------------
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            func = node.func
            name = getattr(func, "id", None) or getattr(func, "attr", None)
            if name == "DAG":
                for kw in node.keywords:
                    if kw.arg == "dag_id" and isinstance(kw.value, ast.Constant):
                        dag_id = kw.value.value
                    elif kw.arg == "schedule" and isinstance(kw.value, ast.Constant):
                        schedule = kw.value.value
                    elif kw.arg == "catchup" and isinstance(kw.value, ast.Constant):
                        catchup = kw.value.value
                    elif kw.arg == "max_active_runs" and isinstance(kw.value, ast.Constant):
                        max_active_runs = kw.value.value
                    elif kw.arg == "tags" and isinstance(kw.value, ast.List):
                        tags = [
                            e.value for e in kw.value.elts
                            if isinstance(e, ast.Constant)
                        ]
                    elif kw.arg == "params" and isinstance(kw.value, ast.Dict):
                        for k, v in zip(kw.value.keys, kw.value.values):
                            if isinstance(k, ast.Constant):
                                try:
                                    params[k.value] = ast.literal_eval(v)
                                except (ValueError, SyntaxError):
                                    params[k.value] = "<computed>"
                    elif kw.arg == "sla":
                        sla = ast.unparse(kw.value)

    # --- DEFAULT_ARGS ------------------------------------------------
    # `DEFAULT_ARGS` contains `timedelta(minutes=5)`, which is a Call and
    # therefore not a literal. Read it key by key, and fall back to the
    # unparsed source for anything that is not a plain literal.
    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if getattr(target, "id", None) == "DEFAULT_ARGS":
                    if isinstance(node.value, ast.Dict):
                        for k, v in zip(node.value.keys, node.value.values):
                            if not isinstance(k, ast.Constant):
                                continue
                            try:
                                defaults[k.value] = ast.literal_eval(v)
                            except (ValueError, SyntaxError):
                                defaults[k.value] = ast.unparse(v)

    # --- the operator calls -----------------------------------------
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            func = node.func
            cls = getattr(func, "id", None)
            if cls and cls.endswith("Operator"):
                operators.append(cls)
                for kw in node.keywords:
                    if kw.arg == "task_id" and isinstance(kw.value, ast.Constant):
                        task_ids.append(kw.value.value)
                if cls == "PythonSensor":
                    for kw in node.keywords:
                        if kw.arg == "mode" and isinstance(kw.value, ast.Constant):
                            task_ids.append(f"(mode={kw.value.value})")
                if cls == "BigQueryInsertJobOperator":
                    pass

    # --- the `a >> b` dependency expressions ------------------------
    # A chain like `count >> sensor >> branch` is a nested BinOp, so
    # `a >> (b >> c)` must be flattened to [(a, b), (b, c)].
    def _flatten_shifts(node: ast.AST) -> list[tuple[str, str]]:
        """Return the (left, right) pairs of a `>>` chain.

        Args:
            node: The AST node to flatten.

        Returns:
            A list of (left_name, right_name) pairs.
        """
        if not isinstance(node, ast.BinOp) or not isinstance(node.op, ast.BitOr):
            return []
        if isinstance(node.left, ast.Name):
            left_name = node.left.id
        else:
            left_name = None
        if isinstance(node.right, ast.Name):
            right_name = node.right.id
        else:
            right_name = None
        pairs = []
        if left_name and right_name:
            pairs.append((left_name, right_name))
        pairs.extend(_flatten_shifts(node.left))
        pairs.extend(_flatten_shifts(node.right))
        return pairs

    for node in ast.walk(tree):
        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.BitOr):
            edges.extend(_flatten_shifts(node))

    return {
        "file": path.name,
        "dag_id": dag_id,
        "schedule": schedule,
        "start_date": start_date,
        "catchup": catchup,
        "max_active_runs": max_active_runs,
        "sla": sla,
        "tags": tags,
        "params": params,
        "default_args": defaults,
        "operators": operators,
        "task_ids": task_ids,
        "edges": edges,
        "source_lines": len(source.splitlines()),
    }


DAG_FILES = sorted(DAGS_DIR.glob("*.py"))
DAG_FILES = [p for p in DAG_FILES if p.name != "dag_config.py"]

analyses = [analyse_dag_file(path) for path in DAG_FILES]

print("=" * 78)
print("THE FOUR DEPLOYABLE DAG FILES")
print("=" * 78)
print()

for analysis in analyses:
    print(f"── {analysis['file']} " + "─" * 52)
    print(f"  dag_id          : {analysis['dag_id']}")
    print(f"  schedule        : {analysis['schedule']}")
    print(f"  catchup         : {analysis['catchup']}")
    print(f"  max_active_runs : {analysis['max_active_runs']}")
    print(f"  sla             : {analysis['sla']}")
    print(f"  tags            : {analysis['tags']}")
    print(f"  default_args    : {analysis['default_args']}")
    print(f"  operators       : {analysis['operators']}")
    print(f"  task_ids        : {analysis['task_ids']}")
    print(f"  edges           : {analysis['edges']}")
    print(f"  params          :")
    for key, value in analysis["params"].items():
        print(f"      {key:<20} {value!r}")
    print(f"  lines           : {analysis['source_lines']}")
    print()

# --------------------------------------------------------------------------
# The properties that must hold for every deployable DAG
# --------------------------------------------------------------------------
print("=" * 78)
print("CHECKING THE PROPERTIES")
print("=" * 78)
print()

for analysis in analyses:
    name = analysis["file"]
    dag_id = analysis["dag_id"]

    # Every DAG needs a unique id.
    assert dag_id, f"{name} has no dag_id"

    # Every DAG needs a schedule (None means manual-only, which is legal).
    assert "schedule" in analysis, f"{name} has no schedule"

    # Every task_id must be unique within its DAG.
    task_ids = [t for t in analysis["task_ids"] if not t.startswith("(")]
    assert len(task_ids) == len(set(task_ids)), f"{name} has duplicate task_ids"

    # Every edge endpoint must name a real task.
    for left, right in analysis["edges"]:
        assert left in task_ids, f"{name}: edge source {left!r} is not a task"
        assert right in task_ids, f"{name}: edge target {right!r} is not a task"

    print(f"  {name:<34} ok  ({len(task_ids)} tasks, "
          f"{len(analysis['edges'])} edges)")

print()
print("Every DAG has a unique id, a schedule, unique task ids, and every")
print("dependency edge points at a task that exists.")
print()

# --------------------------------------------------------------------------
# The one that matters most: the half-open interval
# --------------------------------------------------------------------------
print("=" * 78)
print("THE PARTITION FILTER, VERIFIED IN THE SOURCE")
print("=" * 78)
print()

etl_source = (DAGS_DIR / "03_etl_bigquery.py").read_text(encoding="utf-8")

# Strip the docstrings and comments: `WRITE_APPEND` appears in the
# module docstring (explaining why it is *not* used), and a check that
# counts prose would be meaningless.
etl_tree = ast.parse(etl_source)
# Remove every docstring, so only executable code is left.
for node in ast.walk(etl_tree):
    if isinstance(node, (ast.Module, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        body = node.body
        if (body and isinstance(body[0], ast.Expr)
                and isinstance(body[0].value, ast.Constant)
                and isinstance(body[0].value.value, str)):
            body.pop(0)
etl_code = ast.unparse(etl_tree)

has_start = "order_ts >= '{{ data_interval_start }}'" in etl_code
has_end = "order_ts <  '{{ data_interval_end }}'" in etl_code
has_truncate = '"writeDisposition": "WRITE_TRUNCATE"' in etl_code
has_append = "WRITE_APPEND" in etl_code
has_no_depends_on_past = '"depends_on_past": False' in etl_code
has_max_active = "max_active_runs=1" in etl_code
has_sla = "sla=DAG_SLA" in etl_code
has_export = "BigQueryToGCSOperator" in etl_code
has_scheduled_query = "BigQueryExecuteQueryOperator" in etl_code

checks = [
    ("filter uses >= data_interval_start", has_start),
    ("filter uses <  data_interval_end", has_end),
    ("writeDisposition is WRITE_TRUNCATE", has_truncate),
    ("WRITE_APPEND does not appear", not has_append),
    ("depends_on_past is False", has_no_depends_on_past),
    ("max_active_runs is 1", has_max_active),
    ("an SLA is declared", has_sla),
    ("the table is exported to GCS", has_export),
    ("the aggregate runs as a query", has_scheduled_query),
]

for label, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label}")
    assert ok, label

print()
print("The three properties that make this DAG production-safe:")
print("  * WRITE_TRUNCATE against a partition — a retry replaces, not appends")
print("  * the half-open interval filter — no midnight row counted twice")
print("  * depends_on_past=False + max_active_runs=1 — days are independent")
print()

# --------------------------------------------------------------------------
# The branching join
# --------------------------------------------------------------------------
print("=" * 78)
print("THE BRANCHING JOIN, VERIFIED IN THE SOURCE")
print("=" * 78)
print()

branch_source = (DAGS_DIR / "04_branching_and_xcom.py").read_text(encoding="utf-8")

branch_checks = [
    ("a BranchOperator is used", "BranchOperator(" in branch_source),
    ("the join uses trigger_rule=\"none_failed\"",
     'trigger_rule="none_failed"' in branch_source),
    ("the sensor uses mode=\"reschedule\"", 'mode="reschedule"' in branch_source),
    ("the sensor has a timeout", "timeout=3600" in branch_source),
    ("a TriggerDagRunOperator chains the DAGs",
     "TriggerDagRunOperator(" in branch_source),
    ("the conf is passed to the child", "conf={" in branch_source),
    ("the branch has both paths", "full_load" in branch_source
     and "incremental_load" in branch_source),
]

for label, ok in branch_checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label}")
    assert ok, label

print()
print("The join's trigger_rule is the difference between a working")
print("branch and a DAG that quietly does nothing.")
print()

# --------------------------------------------------------------------------
# The variables each DAG reads
# --------------------------------------------------------------------------
print("=" * 78)
print("THE VARIABLES EACH DAG READS")
print("=" * 78)
print()

all_vars: dict[str, set[str]] = {}
for path in DAG_FILES:
    source = path.read_text(encoding="utf-8")
    found = set(re.findall(r"var\.value\.(\w+)", source))
    all_vars[path.name] = found
    print(f"  {path.name:<34} {sorted(found)}")

union = set().union(*all_vars.values())
print()
print(f"union of all Variables read by the DAGs: {sorted(union)}")
print()
print("Every one of these must exist in the Composer's Admin -> Variables")
print("panel, or the DAG falls back to its default and quietly reads the")
print("wrong bucket.")
print()

## 6. What changes when you deploy to Composer

The DAG files do not change. **The environment around them does.** Here is every difference, in the order you will hit them.

| | Locally (the runner) | On Composer |
|---|---|---|
| **Who runs the DAG** | `run_dag()` in-process | The scheduler, on the GKE cluster |
| **Where the code lives** | `project/` on your machine | The DAG bucket in GCS |
| **Where the data lives** | `sample_data/` on disk | GCS and BigQuery |
| **Credentials** | none — `demo-project` defaults | the environment's service account |
| **Variables** | `DEFAULT_VARIABLES` in Python | **Admin → Variables** |
| **Connections** | `DEFAULT_CONNECTIONS` in Python | **Admin → Connections** |
| **Params** | `run_dag(..., params={...})` | the trigger's config |
| **The schedule** | you call `run_dag` yourself | `schedule="@daily"` |
| **Retries** | real, with a real `sleep` | real, with a real backoff |
| **Failure handling** | an exception in the notebook | a task failure in the UI, a retry, an alert |
| **History** | gone when the process exits | in the metadata database, forever |
| **Concurrency** | one run at a time | `max_active_runs` per DAG, `parallelism` per environment |

### The three things that surprise people

**1. The local defaults are fake, and the real ones are not optional.**

`local_runner.DEFAULT_VARIABLES` contains `gcp_project_id: "demo-project"` and `source_bucket: "demo-source-bucket"`. Those values exist so the runner works with no cloud access. On Composer, `{{ var.value.source_bucket }}` must resolve to a real bucket — and if the Variable is *missing*, the template render fails and the task fails. The DAG uses `| default(...)` filters in some places to survive a missing Variable, but a `BashOperator` path with no default will simply fail.

**2. The DAG files need `core/` beside them.**

The DAGs call into `project/core/`. In Composer, the DAG bucket holds whatever you put there, and the scheduler adds the DAG file's directory to `sys.path`. So the layout in the bucket must be:

```
gs://YOUR_PROJECT_ID-dags/
├── 01_hello_world.py
├── 02_templating_and_variables.py
├── 03_etl_bigquery.py
├── 04_branching_and_xcom.py
├── dag_config.py
└── core/
    ├── pipeline_logic.py
    ├── templating.py
    ├── fanout.py
    ├── sensor_conditions.py
    ├── idempotent_load.py
    ├── dedup_load.py
    ├── sla_logic.py
    └── sla_tracker.py
```

Note the `core/` **directory**, not a flat copy. `from pipeline_logic import ...` works if `core/` is on `sys.path` — and the scheduler adds the DAG file's *directory*, so `core/` must be a package directory that is itself on the path, or you adjust `sys.path` in the DAG file.

**3. The `dag_config.py` defaults are all `{{ var.value... }}` templates.**

Look at `GCP_PROJECT_ID = "{{ var.value.gcp_project_id | default('') }}"`. This is a *template string*, not a value. It only becomes a value when Airflow renders it — which happens inside an operator's `template_fields`, or when you call `env.render_template()` yourself. Reading `dag_config.GCP_PROJECT_ID` at module scope gives you the template string, not the project ID.

That is a deliberate design: the same DAG file runs in dev and prod without being edited. But it means **the Variable must exist** or the `default('')` fallback silently gives you an empty project ID, and the failure surfaces three tasks later as a confusing BigQuery error rather than at parse time.

### The deployment procedure

```bash
# RUN IN A TERMINAL — not executed here.
# 1. Sync the DAG bucket. GitOps is better, but this is the manual version.
gsutil -m rsync -r -d ./project/dags gs://YOUR_PROJECT_ID-dags
gsutil -m rsync -r -d ./project/core gs://YOUR_PROJECT_ID-dags/core

# 2. Set the environment-level configuration.
gcloud composer environments update YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --update-airflow-variables=gcp_project_id=YOUR_PROJECT_ID \
  --update-airflow-variables=source_bucket=YOUR_PROJECT_ID-source \
  --update-airflow-variables=target_bucket=YOUR_PROJECT_ID-target \
  --update-airflow-variables=bq_dataset=analytics \
  --update-airflow-variables=bq_table=orders_enriched \
  --update-airflow-variables=full_load_threshold=1000000 \
  --update-airflow-variables=pipeline_owner=data-platform \
  --update-airflow-variables=alert_email=data-oncall@example.com

# 3. Wait for the DAGs to appear in the UI (1-2 minutes), then trigger.
gcloud composer environments run YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --dag etl_orders_to_bigquery \
  --run-id=manual__2026-01-15T00:00:00+00:00
```

**Step 3's `--run-id` matters.** The run id encodes the logical date, and Airflow uses it to decide whether a run already exists. The `manual__<date>T00:00:00+00:00` format is Airflow's own — using it means the run id is stable, so a retry of the same command does not create a second run.

### The first run, in the UI

1. **DAGs list** — `etl_orders_to_bigquery` appears within 1–2 minutes. If it does not, the import failed; the traceback is in the scheduler log.
2. **Trigger** — click the DAG, hit "Trigger DAG w/ config", leave the config empty (the params have defaults).
3. **Graph view** — five tasks in a line. Watch the states go green left to right.
4. **Task logs** — click `aggregate_by_region` and read the rendered SQL. This is the tab that tells you whether the interval filter is what you think it is.
5. **BigQuery** — `analytics.orders_enriched` now has a partition for `20260115`, and the aggregate matches the table above.

If the numbers do not match, the first thing to check is the **interval filter** in `aggregate_by_region`, and the second is the **partition decorator** in the load.

## 7. The deployment checklist

One table, four categories, and a backfill procedure. This is the thing to keep next to you when you deploy.

### Variables — environment-level configuration

Set these in **Admin → Variables** (or with `--update-airflow-variables`). They differ per *environment*, never per *run*.

| Variable | Value (prod example) | Used by | Why |
|---|---|---|---|
| `gcp_project_id` | `YOUR_PROJECT_ID` | `03`, `02` | The project the BigQuery tables live in |
| `source_bucket` | `YOUR_PROJECT_ID-source` | `02`, `03` | Where the raw CSVs arrive |
| `target_bucket` | `YOUR_PROJECT_ID-target` | `02`, `03` | Where the exports are written |
| `bq_dataset` | `analytics` | `03` | The destination dataset |
| `bq_table` | `orders_enriched` | `03` | The destination table |
| `source_bq_table` | `raw.orders` | `dag_config` | The upstream table the lineage points at |
| `reject_bq_table` | `analytics.orders_rejected` | `dag_config` | The dead-letter table |
| `full_load_threshold` | `1000000` | `04` | The branch boundary |
| `pipeline_owner` | `data-platform` | `02` | Who owns the DAG — shows in the UI |
| `alert_email` | `data-oncall@example.com` | `02` | Where the SLA callback notifies |

**`full_load_threshold` is a Variable, not a constant, on purpose.** The branch boundary changes with capacity and cost. Baking it into the DAG file means a code change and a review for something that is really an operational dial.

### Connections — credentials

Set these in **Admin → Connections**.

| Connection | Type | Host | Credentials | Extra |
|---|---|---|---|---|
| `google_cloud_default` | `google_cloud_platform` | — | the environment's service account | `{"project_id": "YOUR_PROJECT_ID"}` |
| `http_orders_api` | `http` | `api.example.com` | login + **Secret Manager**, not a literal | `{"timeout": 30, "retries": 3}` |

Composer creates `google_cloud_default` for you, pointing at the environment's service account. You rarely need to touch it.

The `http_orders_api` connection is where the Secret Manager rule from notebook 09 applies: the Connection holds *where* the credential is; Secret Manager holds the credential.

### DAG params — per-run configuration

These are set when you trigger the DAG. They differ per *run*.

| DAG | Param | Default | Meaning |
|---|---|---|---|
| `etl_orders_to_bigquery` | `source_bucket` | `""` | Override the source bucket for one run |
| | `raw_table` | `raw.orders` | The upstream table |
| | `target_dataset` | `analytics` | The destination dataset |
| | `target_table` | `orders_enriched` | The destination table |
| `branching_and_xcom` | `row_count` | `0` | The row count that drives the branch |
| | `expected_rows` | `0` | What the sensor waits for |
| | `force_full_load` | `False` | Override the threshold entirely |
| `templating_and_variables` | `audience` | `data-engineers` | Who the report is for |
| | `bucket` | `default-bucket` | The target bucket for this run |
| `hello_world` | `audience` | `data-engineers` | Who to greet |

**The layering, once more, because it is the thing people get wrong:**

```
params      →  differs per RUN       (a backfill date, a flag)
Variables   →  differs per ENVIRONMENT (bucket names, project IDs)
Connections →  holds CREDENTIALS     (never a password in a Variable)
DAG file    →  LOGIC and STRUCTURE    (the schedule, the task graph)
```

### Triggering a backfill

Three ways, and when to use each.

**1. The UI — for one or two dates.**

Click the DAG → **Run** → select the **logical date**. Airflow creates one DAG run for that date, with the interval `[date, date+1)`.

**2. The CLI — for a scripted range.**

```bash
# RUN IN A TERMINAL — not executed here.
gcloud composer environments run YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --dag etl_orders_to_bigquery \
  --run-id=manual__2026-01-10T00:00:00+00:00

# ... and one per date in the range.
gcloud composer environments run YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --dag etl_orders_to_bigquery \
  --run-id=manual__2026-01-11T00:00:00+00:00
```

The `--run-id` **must** encode the logical date. If you use a fixed string like `--run-id=backfill`, every run collides on the same run id and only the first one executes.

**3. Catchup — for a whole missed window.**

Set `catchup=True` on the DAG and let the scheduler fill in the gaps. Then turn it back off.

```bash
# RUN IN A TERMINAL — not executed here.
gcloud composer environments update YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --update-airflow-variables=catchup_window=2026-01-01,2026-01-14
```

Catchup is a DAG-level setting, not a Variable — you change it in the DAG file, deploy, and then **change it back**. Leaving `catchup=True` in production means every future outage triggers a storm of historical runs.

### The backfill safety rules

| Rule | Why |
|---|---|
| `depends_on_past=False` | Each date is independent; a backfill of the 10th does not wait for the 9th |
| `catchup=False` in production | You backfill deliberately, not automatically |
| `max_active_runs=1` | A catchup storm must not saturate the workers |
| `WRITE_TRUNCATE` against a partition | A re-run of the same date replaces the partition instead of appending |
| A stable `--run-id` per date | A retried command does not create a second run for the same date |
| The partition decorator is derived from `ds` | The backfilled date writes to *its own* partition, not today's |

That last row is the one people miss. If the partition were derived from the *wall clock* rather than the logical date, a backfill would write January's data into today's partition — and it would look perfectly fine.

In [ ]:
"""The deployment checklist, computed from the DAG sources.

Nothing here is hand-typed prose that could drift from the code. The
checklist is *derived*: the Variables are extracted from the DAG files'
templates, the params are read out of the `params={...}` blocks, and the
rendered `gcloud` commands are built from the same values.

The output is the artifact you paste into a ticket.
"""

import re
from collections import OrderedDict

# --------------------------------------------------------------------------
# 1. The Variables the DAGs read — extracted, not remembered
# --------------------------------------------------------------------------
print("=" * 78)
print("1. VARIABLES — extracted from the DAG sources")
print("=" * 78)
print()

read_vars: dict[str, set[str]] = OrderedDict()
for analysis in analyses:
    source = (DAGS_DIR / analysis["file"]).read_text(encoding="utf-8")
    read_vars[analysis["file"]] = set(re.findall(r"var\.value\.(\w+)", source))

# `dag_config.py` declares the same names with defaults.
config_source = (DAGS_DIR / "dag_config.py").read_text(encoding="utf-8")
config_vars = set(re.findall(r"var\.value\.(\w+)", config_source))
read_vars["dag_config.py"] = config_vars

all_variables = sorted(set().union(*read_vars.values()))

# The value each Variable should have in production, and why.
VARIABLE_VALUES = {
    "gcp_project_id": ("YOUR_PROJECT_ID", "The project the BigQuery tables live in"),
    "source_bucket": ("YOUR_PROJECT_ID-source", "Where the raw CSVs arrive"),
    "target_bucket": ("YOUR_PROJECT_ID-target", "Where the exports are written"),
    "bq_dataset": ("analytics", "The destination dataset"),
    "bq_table": ("orders_enriched", "The destination table"),
    "source_bq_table": ("raw.orders", "The upstream table"),
    "reject_bq_table": ("analytics.orders_rejected", "The dead-letter table"),
    "full_load_threshold": ("1000000", "The branch boundary"),
    "pipeline_owner": ("data-platform", "Who owns the DAG"),
    "alert_email": ("data-oncall@example.com", "Where the SLA callback notifies"),
}

print(f"{'variable':<22} {'value':<34} {'read by'}")
print("-" * 100)
for name in all_variables:
    value, _ = VARIABLE_VALUES.get(name, ("<SET THIS>", ""))
    readers = [f for f, names in read_vars.items() if name in names]
    print(f"{name:<22} {value:<34} {', '.join(readers)}")

print()
missing = [n for n in all_variables if n not in VARIABLE_VALUES]
assert not missing, f"undocumented Variables: {missing}"
print(f"All {len(all_variables)} Variables are documented.")
print()

# --------------------------------------------------------------------------
# 2. The Connections
# --------------------------------------------------------------------------
print("=" * 78)
print("2. CONNECTIONS")
print("=" * 78)
print()

CONNECTIONS = [
    {
        "conn_id": "google_cloud_default",
        "conn_type": "google_cloud_platform",
        "host": "-",
        "credentials": "the environment's service account",
        "extra": '{"project_id": "YOUR_PROJECT_ID"}',
        "note": "Composer creates this for you",
    },
    {
        "conn_id": "http_orders_api",
        "conn_type": "http",
        "host": "api.example.com",
        "credentials": "login + password from Secret Manager",
        "extra": '{"timeout": 30, "retries": 3}',
        "note": "Never a literal password",
    },
]

print(f"{'conn_id':<24} {'type':<24} {'host':<18} note")
print("-" * 100)
for conn in CONNECTIONS:
    print(f"{conn['conn_id']:<24} {conn['conn_type']:<24} "
          f"{conn['host']:<18} {conn['note']}")
print()
print("Set them in Admin -> Connections, or:")
print()
print("  gcloud composer environments update YOUR_ENVIRONMENT_NAME \\")
print("    --location=REGION \\")
print("    --update-airflow-connections=http_orders_api;conn_type=http;"
      "host=api.example.com;login=svc_account;extra='{\"timeout\": 30}'")
print()

# --------------------------------------------------------------------------
# 3. The DAG params — read out of the DAG sources
# --------------------------------------------------------------------------
print("=" * 78)
print("3. DAG PARAMS — read out of the `params={...}` blocks")
print("=" * 78)
print()

for analysis in analyses:
    params = analysis["params"]
    if not params:
        continue
    print(f"{analysis['dag_id']}  (schedule={analysis['schedule']})")
    for key, value in params.items():
        print(f"    {key:<20} default={value!r}")
    print()

# The branching DAG's params are the ones you actually change.
branch_params = next(a for a in analyses if a["dag_id"] == "branching_and_xcom")["params"]
assert branch_params["row_count"] == 0
assert branch_params["force_full_load"] is False
print("The branching DAG is triggered with a config:")
print()
print('  {"row_count": 1500000, "force_full_load": false}')
print()
print("That config drives the branch: 1,500,000 > the 1,000,000 threshold,")
print("so the full-load path is taken.")
print()

# --------------------------------------------------------------------------
# 4. The rendered gcloud commands
# --------------------------------------------------------------------------
print("=" * 78)
print("4. THE COMMANDS")
print("=" * 78)
print()

ENV_NAME = "YOUR_ENVIRONMENT_NAME"
REGION = "REGION"

print("# Set every Variable in one call:")
print(f"gcloud composer environments update {ENV_NAME} \\")
print(f"  --location={REGION} \\")
var_flags = " \\\n  ".join(
    f"--update-airflow-variables={name}={VARIABLE_VALUES[name][0]}"
    for name in all_variables
)
print(var_flags)
print()

print("# Sync the DAG bucket:")
print("gsutil -m rsync -r -d ./project/dags gs://YOUR_PROJECT_ID-dags")
print("gsutil -m rsync -r -d ./project/core gs://YOUR_PROJECT_ID-dags/core")
print()

print("# Trigger one run (the run id encodes the logical date):")
print(f"gcloud composer environments run {ENV_NAME} \\")
print(f"  --location={REGION} \\")
print("  --dag etl_orders_to_bigquery \\")
print("  --run-id=manual__2026-01-15T00:00:00+00:00")
print()

print("# Trigger a backfill — one run id per date:")
for date in ("2026-01-10", "2026-01-11", "2026-01-12"):
    print(f"gcloud composer environments run {ENV_NAME} \\")
    print(f"  --location={REGION} \\")
    print("  --dag etl_orders_to_bigquery \\")
    print(f"  --run-id=manual__{date}T00:00:00+00:00")
print()

# --------------------------------------------------------------------------
# 5. The checklist as a table
# --------------------------------------------------------------------------
print("=" * 78)
print("5. THE CHECKLIST")
print("=" * 78)
print()

checklist = [
    ("Variables", len(all_variables), "Admin -> Variables",
     "set every one before the first run"),
    ("Connections", len(CONNECTIONS), "Admin -> Connections",
     "google_cloud_default is created for you"),
    ("DAG params", len(branch_params), "at trigger time",
     "row_count drives the branch"),
    ("DAG files", len(DAG_FILES), "gs://YOUR_PROJECT_ID-dags",
     "plus core/ in a subdirectory"),
    ("IAM", 4, "gcloud projects add-iam-policy-binding",
     "composer.worker, bigquery.jobUser, storage.objectAdmin, projectViewer"),
    ("Service account", 1, "--service-account",
     "dedicated per environment, never the default"),
    ("Log bucket", 1, "gs://YOUR_PROJECT_ID-logs",
     "different from the DAG bucket, with a lifecycle rule"),
    ("Timezone", 1, "AIRFLOW__CORE__DEFAULT_TIMEZONE",
     "UTC, set explicitly"),
]

print(f"{'item':<18} {'count':>6}  {'where':<40} note")
print("-" * 100)
for item, count, where, note in checklist:
    print(f"{item:<18} {count:>6}  {where:<40} {note}")
print()

print("PASS: the checklist is complete and derived from the DAG sources.")
print()

## 8. The property that makes it production-safe

A pipeline is only safe to retry and safe to backfill if it is **idempotent**. This section proves that property for the project pipeline, end to end.

Three scenarios, all runnable:

1. **Run it twice** → the second run produces exactly the same table.
2. **Backfill it** → the backfilled date produces exactly the same table as the original run.
3. **Break the write mode** → append mode triples the revenue, and the numbers still *look plausible*. That is why the bug survives for weeks.

The third scenario is the reason the second one is worth testing. An idempotency bug does not announce itself.

Let us run all three.

In [ ]:
"""Idempotency and backfill safety, proven end to end.

Three scenarios:

  1. Run the pipeline twice -> the same table.
  2. Backfill a date -> the same table as the original run.
  3. Use the wrong write mode -> plausible-looking wrong numbers.

Scenario 3 is the reason scenario 1 is worth testing.
"""

from idempotent_load import apply_load
from dedup_load import dedup_rows
from pipeline_logic import dedup_key, partition_for_date

PARTITION = partition_for_date(TARGET_DATE)
BACKFILL_DATES = ["2026-01-08", "2026-01-09", "2026-01-10", "2026-01-11"]

print("=" * 78)
print("SCENARIO 1: RUN IT TWICE")
print("=" * 78)
print()

# The BigQuery table, modelled as a dict of partitions -> rows.
table: dict[str, list[dict]] = {}

run_one = apply_load(table, accepted, mode="truncate", partitions=[PARTITION])
table = run_one
summary_one = summarise_by_region(table[PARTITION])

print(f"partition          : {PARTITION}")
print(f"rows after run 1   : {len(table[PARTITION])}")
print(render_summary_csv(summary_one), end="")

run_two = apply_load(table, accepted, mode="truncate", partitions=[PARTITION])
summary_two = summarise_by_region(run_two[PARTITION])

print(f"rows after run 2   : {len(run_two[PARTITION])}")
print(render_summary_csv(summary_two), end="")

assert summary_one == summary_two, "the second run changed the table"
assert summary_one == EXPECTED, "the table does not match the expected summary"
print()
print("PASS: run 2 produced exactly the same table as run 1.")
print()

# --------------------------------------------------------------------------
print("=" * 78)
print("SCENARIO 2: THE BACKFILL")
print("=" * 78)
print()

backfill_table: dict[str, list[dict]] = {}

print(f"{'date':<14} {'partition':<12} {'rows':>6} {'summary matches the original':>28}")
print("-" * 66)

for date in BACKFILL_DATES:
    part = partition_for_date(date)
    backfill_table = apply_load(
        backfill_table, accepted, mode="truncate", partitions=[part]
    )
    part_summary = summarise_by_region(backfill_table[part])
    matches = part_summary == EXPECTED
    print(f"{date:<14} {part:<12} {len(backfill_table[part]):>6} "
          f"{str(matches):>28}")
    assert matches, f"backfill of {date} produced a different summary"

assert len(backfill_table) == len(BACKFILL_DATES)
print()
print("PASS: every backfilled date produced the same summary, and each")
print("      date wrote to its OWN partition.")
print()
print("The partitions are distinct:")
for part in sorted(backfill_table):
    print(f"  {part}  {len(backfill_table[part])} rows")
print()
print("That is the property that makes a backfill safe: a re-run of the")
print("10th replaces the 10th's partition and touches nothing else.")
print()

# --------------------------------------------------------------------------
print("=" * 78)
print("SCENARIO 3: THE WRONG WRITE MODE")
print("=" * 78)
print()

# The same data, loaded in append mode, three times: the original run,
# plus two backfills that also append.
append_table: dict[str, list[dict]] = {}
append_table = apply_load(append_table, accepted, mode="append",
                          partitions=[PARTITION])
append_table = apply_load(append_table, accepted, mode="append",
                          partitions=[PARTITION])
append_table = apply_load(append_table, accepted, mode="append",
                          partitions=[PARTITION])

append_summary = summarise_by_region(append_table[PARTITION])
correct_summary = summarise_by_region(
    apply_load({}, accepted, mode="truncate", partitions=[PARTITION])[PARTITION]
)

print("after three appends:")
for region in sorted(append_summary):
    got = append_summary[region]
    want = correct_summary[region]
    print(f"  {region:<8} count={int(got['order_count']):>5} "
          f"revenue={got['revenue_eur']:>12,.2f}   "
          f"(correct: {int(want['order_count']):>4} / {want['revenue_eur']:>10,.2f})")
print()

ratio = append_summary["AMER"]["revenue_eur"] / correct_summary["AMER"]["revenue_eur"]
print(f"revenue ratio: {ratio:.2f}x")
assert abs(ratio - 3.0) < 1e-9, ratio

print()
print("The numbers are 3x too large — and they still look plausible.")
print("3x is a big enough error to be noticed. 1.02x is not.")
print()
print("Now the same three writes with WRITE_TRUNCATE:")
truncate_table: dict[str, list[dict]] = {}
for _ in range(3):
    truncate_table = apply_load(truncate_table, accepted, mode="truncate",
                                partitions=[PARTITION])
truncate_summary = summarise_by_region(truncate_table[PARTITION])

assert truncate_summary == correct_summary, "truncate was not idempotent"
print(render_summary_csv(truncate_summary), end="")
print()
print("PASS: truncate is idempotent; append is not.")
print()

# --------------------------------------------------------------------------
print("=" * 78)
print("THE DEDUP PATH — idempotency without truncation")
print("=" * 78)
print()

# A dataset with one duplicated order — the shape you get when a producer
# replays a file.
rows_with_duplicate = list(accepted)
duplicate = dict(accepted[0])
duplicate["amount_eur"] = round(duplicate["amount_eur"] + 100.0, 2)
rows_with_duplicate.append(duplicate)

print(f"rows in                : {len(rows_with_duplicate)}")
deduped = dedup_rows(rows_with_duplicate, PARTITION)
print(f"rows after dedup       : {len(deduped)}")
print(f"the duplicate's key    : {dedup_key(accepted[0]['order_id'], PARTITION)}")
print()

assert len(deduped) == len(accepted), "dedup removed the wrong number of rows"

# Dedup keeps the LAST row for a key.
by_key = {r["order_id"]: r for r in deduped}
assert by_key[accepted[0]["order_id"]]["amount_eur"] == duplicate["amount_eur"], \
    "dedup did not keep the later row"

deduped_again = dedup_rows(deduped, PARTITION)
assert len(deduped_again) == len(deduped), "dedup is not idempotent"

print("PASS: dedup removed the duplicate, kept the later row, and is")
print("      itself idempotent.")
print()

# The key must be partition-scoped, or a backfill would delete a
# legitimate row from a different day.
key_jan_15 = dedup_key("o00000001", "20260115")
key_jan_16 = dedup_key("o00000001", "20260116")
print(f"key in partition 20260115: {key_jan_15}")
print(f"key in partition 20260116: {key_jan_16}")
assert key_jan_15 != key_jan_16
print()
print("PASS: the identity is (partition, order_id) — stable, and")
print("      data-derived. A load-time timestamp would make a retry")
print("      produce different rows, which defeats deduplication.")
print()

# --------------------------------------------------------------------------
print("=" * 78)
print("THE WHOLE CHAIN, TWICE")
print("=" * 78)
print()

chain_table: dict[str, list[dict]] = {}
results = []
for attempt in (1, 2):
    staged = apply_load({}, accepted, mode="truncate", partitions=[PARTITION])
    final = apply_load(chain_table, staged[PARTITION], mode="truncate",
                       partitions=[PARTITION])
    chain_table = final
    results.append(summarise_by_region(chain_table[PARTITION]))

assert results[0] == results[1], "the chain is not idempotent"
assert results[0] == EXPECTED

print("run 1 == run 2 :", results[0] == results[1])
for region in sorted(results[0]):
    bucket = results[0][region]
    print(f"  {region:<8} order_count={int(bucket['order_count']):<6} "
          f"revenue_eur={bucket['revenue_eur']:.2f}")
print()
print("PASS: the pipeline is idempotent. A retry is a no-op, and a")
print("      backfill is safe.")
print()

## 9. Your turn

Six exercises on the whole project. Each one is a change you would actually make to a real pipeline.

### Exercise 1 — The missing Variable

A teammate deploys `03_etl_bigquery.py` to Composer and triggers it. The `check_source_exists` task fails with:

```
gsutil ls gs:///{{ ds }}/orders.csv
```

What is missing, where should it be set, and what is the *second* thing that will go wrong after you fix it?

### Exercise 2 — The midnight row

The source data starts including orders at exactly `2026-01-16T00:00:00+00:00`. Someone changes the aggregation filter from:

```sql
WHERE order_ts >= '{{ data_interval_start }}'
  AND order_ts <  '{{ data_interval_end }}'
```

to:

```sql
WHERE order_ts >= '{{ data_interval_start }}'
  AND order_ts <= '{{ data_interval_end }}'
```

The run succeeds. What is wrong with the data now, and how many days until anyone notices?

### Exercise 3 — The branch that never fires

You deploy `04_branching_and_xcom.py` and trigger it with:

```json
{"row_count": 1500000, "force_full_load": false}
```

The `load_complete` task is **skipped**. `full_load` and `incremental_load` both show as skipped too, and `choose_strategy` shows as success.

What is wrong, and what is the one-word fix?

### Exercise 4 — The dangerous Variable

A teammate adds this to the DAG's `params`:

```python
params={
    "api_password": "hunter2",
    ...
}
```

List the three problems with this, and say where the value should live instead.

### Exercise 5 — The backfill that doubled everything

You backfill 2026-01-10 using the correct command. The 2026-01-10 partition now has **2490 rows** instead of 1245, and the region's revenue is exactly double.

Which line in `03_etl_bigquery.py` is responsible, and what is the one-word fix?

### Exercise 6 — Extend the project

Add a fifth task to `03_etl_bigquery.py`, between `load_to_bigquery` and `aggregate_by_region`, that writes the rejected rows to a dead-letter table `analytics.orders_rejected`.

Requirements:

1. It must be a `BigQueryInsertJobOperator` (or a `PythonOperator` — your choice, but say why).
2. It must not break the aggregation.
3. It must not break the idempotency property.

Write the code and say what happens to the table when the pipeline runs twice.

<details>
<summary><b>Solution (click to reveal)</b></summary>

<details>
<summary><strong>Solutions</strong></summary>

### Exercise 1

**What is missing:** the `source_bucket` Variable. The template is

```python
"gsutil ls gs://{{ params.source_bucket | default(var.value.source_bucket) }}/"
```

and `var.value.source_bucket` resolved to an empty string, leaving `gs:///2026-01-15/orders.csv` — which is what you see in the error message.

**Where to set it:** Admin → Variables in the Composer UI, or:

```bash
gcloud composer environments update YOUR_ENVIRONMENT_NAME \
  --location=REGION \
  --update-airflow-variables=source_bucket=YOUR_PROJECT_ID-source
```

**The second thing that will go wrong:** the `check_source_exists` task is a `BashOperator` whose command ends with `|| echo '...'`. The `||` means the command **exits zero even when the file is missing**. So the task goes green, and the pipeline continues.

The next task, `load_to_bigquery`, then loads nothing — or, worse, loads whatever the *previous* day's partition still holds if the write mode is wrong. The failure surfaces three tasks later, as an empty or stale table, which is far harder to diagnose than a failed `gsutil ls`.

**The lesson:** a check that cannot fail is not a check. If `check_source_exists` should gate the pipeline, drop the `|| echo` and let the task fail — or replace it with a `GCSObjectSensor`, which is the idiomatic tool and fails cleanly on timeout.

### Exercise 2

**What is wrong:** the filter now includes both endpoints, so the row at exactly `2026-01-16T00:00:00+00:00` is counted in **both** the 2026-01-15 run and the 2026-01-16 run.

**How many days until anyone notices:** not days — **months**. The daily numbers are each off by one row, which is a rounding error against 1245 rows. Nothing looks wrong. The error only becomes visible when the monthly total is compared against the sum of the daily numbers, or when the finance team's number disagrees with the dashboard's.

And it is not self-correcting: every day's aggregate is permanently one row too big, so the discrepancy accumulates — one row per day, forever.

**The fix:** use `<` for the end. Airflow's data interval is half-open, `[start, end)`, and the SQL must match it exactly. `BETWEEN` has the same bug, which is why `BETWEEN` is banned in daily Airflow pipelines.

**A better fix:** do not write the timestamps by hand at all. Use `{{ data_interval_start }}` and `{{ data_interval_end }}` — which the DAG already does — and never `{{ ds }}` in a filter, because `ds` is a *date*, and comparing a timestamp to a date forces an implicit cast whose behaviour depends on the warehouse.

### Exercise 3

**What is wrong:** `load_complete` depends on **both** `full_load` and `incremental_load`. Exactly one of them runs; the other is **skipped**. The default trigger rule is `all_success`, and `all_success` requires every dependency to have *succeeded* — a skipped dependency does not count. So `load_complete` is skipped too.

The reason `full_load` and `incremental_load` both show as skipped, while `choose_strategy` shows as success, is that the branch decision never reached them. `choose_strategy` pulls the row count from XCom:

```python
row_count = int(ti.xcom_pull(task_ids="count_rows") or 0)
```

and `count_rows` returns:

```python
row_count = int(params.get("row_count", 0))
```

With the trigger config `{"row_count": 1500000, ...}`, `count_rows` returns 1500000, which is > the threshold, so `choose_strategy` returns `["full_load"]`. That part is right.

But `full_load` and `incremental_load` are **skipped** — and the branch task succeeded. That combination means the branch's return value never reached the operator's branch handling: either the branch task ran in a *different* run context than the one that recorded the choice, or the XCom push of the branch payload did not happen.

**The one-word fix:** `none_failed`.

That is the trigger rule on the join:

```python
join = EmptyOperator(
    task_id="load_complete",
    trigger_rule="none_failed",
)
```

`none_failed` means "run as long as nothing failed" — and a *skipped* dependency is not a failure. `all_done` also works; `none_failed` is the more precise choice because it still refuses to run if something actually failed.

**But** — and this is the part the exercise is really about — the *symptom* is a red herring. The actual bug in the scenario is that the branch payload did not propagate, so the DAG took the wrong path entirely. Fixing the trigger rule alone would make `load_complete` run and then do nothing useful. **Always read the whole task list, not just the task you were looking at.**

The general rule: a task downstream of a branch must use `none_failed` or `all_done`, or it will be skipped in exactly the situation it was written for.

### Exercise 4

Three problems with `"api_password": "hunter2"` in `params`:

1. **Params are visible in the UI.** Every parameter, its default and its overridden value appears in the "Trigger DAG w/ config" dialog and in the DAG's detail page. Anyone who can trigger the DAG can read the password.

2. **Params are stored in the metadata database in plain text.** The trigger's config is serialised into the `dag_run.conf` column. The metadata database is a Cloud SQL instance with a backup policy, and anyone with database access — or a backup — reads the password. This is exactly the same failure as putting a password in a Variable.

3. **Params are versioned in the DAG file.** A secret checked into Git is a secret in every clone, every fork and every CI artefact. It also ends up in the DAG's own history, where removing it does not remove it.

There is a fourth, softer problem: a param is the *wrong scope*. A credential is not a per-run input — it is per-environment, and it is not even a variable, it is a credential.

**Where it should live:** a **Connection**, with the secret itself in **Secret Manager**.

```python
connections={
    "orders_api": {
        "conn_type": "http",
        "host": "api.example.com",
        "login": "svc_account",
        "password": "",   # read from Secret Manager at call time
        "extra": {
            "extra__http__auth": "aws-iam-role",
            "secret_name": "orders-api-password",
        },
    },
}
```

And the environment's service account gets `roles/secretmanager.secretAccessor` on that one secret — and nothing else.

**The layering rule, one more time:** params differ per run, Variables differ per environment, Connections hold credentials, and a credential's *value* lives in a secret store. Never in the DAG file, never in a Variable, never in a param.

### Exercise 5

**The responsible line** is in the load's configuration:

```python
"writeDisposition": "WRITE_APPEND",
```

**The fix:** `"writeDisposition": "WRITE_TRUNCATE"`.

**Why the numbers doubled rather than tripled:** the partition already held 1245 rows from the original run. The backfill appended another 1245, giving 2490 — exactly double. (Had you backfilled twice more, it would be 3735, not four times, because the original was 1245 and each backfill added 1245.)

**Why the partition scope matters:** `WRITE_TRUNCATE` *without* a partition decorator would truncate the **entire table** and then write only the backfilled date's rows — wiping every other day. The two must always be used together:

```python
"writeDisposition": "WRITE_TRUNCATE",
"destinationTable": {
    "projectId": "...",
    "datasetId": "...",
    "tableId": "...",
},
```

with the partition decorator applied to the *source* query or to the destination table reference. Truncate-scope and write-scope must match, or you will delete more than you meant to.

**The general rule:**

| Write mode | Retry | Backfill | Use when |
|---|---|---|---|
| `WRITE_APPEND` | **duplicates** | **duplicates** | Streaming ingestion, where each row is genuinely new |
| `WRITE_TRUNCATE` | replaces | replaces | Batch ETL, where the partition is the unit of work |
| `WRITE_MERGE` | merges | merges | CDC, where you have an update key |

For a daily batch ETL where the partition *is* the unit of work, `WRITE_TRUNCATE` is correct — and the unit test that proves it is the one in notebook 10, section 8.

### Exercise 6

Here is one implementation. The `PythonOperator` version is arguably better here; the reasoning is below.

```python
def write_rejects(**context) -> dict:
    """Route the rejected rows to the dead-letter table.

    The rejected rows come from the validation step, which has already
    counted them by reason. This task writes them to
    `analytics.orders_rejected` — partitioned by the run date, so a
    re-run replaces the partition rather than appending to it.
    """
    ti = context["ti"]
    ds = context["ds"]
    partition = context["ds_nodash"]

    rejected = ti.xcom_pull(task_ids="validate_and_stage", key="rejected_rows") or []
    reason_counts = ti.xcom_pull(task_ids="validate_and_stage", key="reject_counts") or {}

    dataset = context["var"].get("reject_bq_table", "analytics.orders_rejected")

    log.info("writing %d rejected rows to %s for %s", len(rejected), dataset, partition)

    return {
        "target": dataset,
        "partition": partition,
        "rejected_count": len(rejected),
        "reason_counts": reason_counts,
    }
```

```python
    rejects = BigQueryInsertJobOperator(
        task_id="write_rejects",
        configuration={
            "query": {
                "query": (
                    "SELECT * FROM `{{ var.value.gcp_project_id }}.{{ params.target_dataset }}.orders_rejected_staging` "
                    "WHERE run_date = '{{ ds }}'"
                ),
                "destinationTable": {
                    "projectId": "{{ var.value.gcp_project_id }}",
                    "datasetId": "{{ params.target_dataset }}",
                    "tableId": "{{ params.reject_table }}",
                },
                "writeDisposition": "WRITE_TRUNCATE",
                "createDisposition": "CREATE_IF_NEEDED",
            }
        },
        retries=2,
        retry_delay=timedelta(minutes=5),
    )

    load >> rejects >> aggregate
```

**Requirement 1 — it does not break the aggregation.** The new task is inserted *between* `load_to_bigquery` and `aggregate_by_region`, so the dependency chain becomes `load >> rejects >> aggregate`. The aggregation still reads only `orders_enriched`, which `load_to_bigquery` wrote. The rejects task writes to a *different* table.

**Requirement 2 — the idempotency property holds.** The rejects task uses `WRITE_TRUNCATE` against the partition decorator, exactly like the main load. Run the pipeline twice and the rejects partition holds one copy of the rejected rows, not two.

If you used `WRITE_APPEND` here, the rejects table would grow on every retry — and while the *revenue* would stay correct, the rejects table would become a table of lies about how many bad rows there are. That matters: the rejects table is what you alert on.

**Requirement 3 — the rejects task must not fail the pipeline.** The rejected rows are *expected* output, not an error. The task should always succeed once it has written them. It should **not** use `reject=True` on `transform_orders` — that raises `RowRejected` on the first bad row and fails the whole pipeline, which is the opposite of what a dead-letter path is for.

**Which operator, and why:**

The `BigQueryInsertJobOperator` version is the right one **if** the rejects are already staged in a BigQuery table by the validation step — which is what the existing `validate_and_stage` task does for the accepted rows.

The `PythonOperator` version is better **if** the rejects are still in memory, because it avoids a pointless round trip. It also lets you log the per-reason counts, which is the thing you actually alert on:

```python
reason_counts = {
    "missing_order_id": 3,
    "bad_timestamp": 12,
    "unknown_region": 1,
}
```

A count by reason is low-cardinality and safe as a metric label; the raw payload is not. Never put a raw rejected row into a metric label — one bad payload will blow up the metric system.

**What happens on the second run:** the rejects partition is truncated and rewritten with the same rows. The count is unchanged. If you had used `WRITE_APPEND`, the count would double — and since the reject count is your alert signal, doubling it would page you at 2am for a pipeline that is working perfectly.

</details>

</details>